# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library. All record sets, fields, and columns are referenced by their `@id` as per FAIR data practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Display dataset name and description
print("Dataset title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)

# Display available record sets
print("\nAvailable record sets (by @id):")
for rs in dataset.metadata.recordSets:
    print(f"- {rs['@id']} : {rs.get('name', 'No name')} ({rs.get('description', 'No description')})")

## 2. Data Overview
Review available record sets, their fields, and columns—including their `@id`s. This step helps identify how the data is structured and which identifiers to use for extraction.

In [ ]:
# List all record sets, fields, and columns by @id
for record_set in dataset.metadata.recordSets:
    rs_id = record_set['@id']
    rs_name = record_set.get('name', '[Unnamed]')
    print(f"\nRecordSet @id: {rs_id} | Name: {rs_name}")
    if 'fields' in record_set:
        for field in record_set['fields']:
            f_id = field['@id']
            f_name = field.get('name', '[Unnamed field]')
            print(f"  Field @id: {f_id} | Name: {f_name}")
            if 'columns' in field:
                for col in field['columns']:
                    c_id = col['@id']
                    c_name = col.get('name', '[Unnamed column]')
                    print(f"    Column @id: {c_id} | Name: {c_name}")

## 3. Data Extraction
Load data from each record set using its `@id`. The extracted data is placed in DataFrames, using the unique `@id` of each entity.

In [ ]:
# Extract all available record set @ids
record_sets_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
dataframes = {}

for rs_id in record_sets_ids:
    print(f"\nLoading RecordSet @id: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns in {rs_id}: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
To explore and process the dataset, select a numeric field (referenced by its `@id`) and demonstrate filtering, normalization, and grouping.

In [ ]:
# Example of selecting a record set and numeric field for EDA
# Modify these IDs if you want to target other fields/record sets found in the previous overview step.

record_set_id = record_sets_ids[0]  # Take the first record set for illustration
df = dataframes[record_set_id]

print(f"Using RecordSet @id: {record_set_id}")
fields_info = dataset.metadata.recordSets[0]['fields']
numeric_field_id = None
group_field_id = None
for field in fields_info:
    if field.get('dataType') in ['schema:Float', 'schema:Integer', 'schema:Number']:
        numeric_field_id = field['@id']
        break
for field in fields_info:
    if field.get('dataType') == 'schema:Text':
        group_field_id = field['@id']
        break

# Check if numeric_field_id is present
if numeric_field_id is None:
    print("No numeric field found for EDA.")
else:
    print(f"Numeric Field @id: {numeric_field_id}")
    # Filter by a threshold
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Grouping
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Numeric field {numeric_field_id} not present in the dataframe columns.")

## 5. Visualization
Visualize distributions or relationships using the extracted data. Example plot: histogram of the selected numeric field (by `@id`).

In [ ]:
# Visualization example: plot a histogram for a numeric field referenced by @id
if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    df[numeric_field_id].dropna().hist(bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

# If grouped data exists, plot bar chart
if group_field_id and group_field_id in df.columns and numeric_field_id in df.columns:
    grouped = df.groupby(group_field_id)[numeric_field_id].mean()
    grouped.plot(kind='bar', figsize=(10,5))
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, process, and visualize a FAIR² survey dataset using `mlcroissant`. All data entities—including record sets and fields—were referenced via their `@id` to ensure FAIR compliance. We explored data structure, filtered and normalized numeric fields, grouped records, and visualized outcomes. For more sophisticated analysis, adapt the EDA section to specific research questions using the full set of available field and column `@id`s.